# SynBioCrow 2.3 — Galaxy Held-Out Benchmark: Sealed Predictions\nRuns the 64 structurally executable pathways from the 65-path held-out Galaxy benchmark. The unresolved target remains in the final benchmark denominator as MAPPING_LIMITED. Literature reaction truth is not passed to the generators. Outputs checkpoint to Drive and resume per target/arm.\n

In [ ]:
import os,sys,subprocess,shutil,time,json,gzip,urllib.request,importlib,sqlite3,hashlib,zipfile
from pathlib import Path
from google.colab import drive, files

T0=time.time()
def log(msg):
    print(f"[GALAXY BENCH SEALED] elapsed {(time.time()-T0)/60:6.1f} min | {msg}",flush=True)

def run(cmd,*,cwd=None,timeout=3600,check=True,stream=False):
    print("$"," ".join(map(str,cmd)),flush=True)
    if stream:
        p=subprocess.Popen(list(map(str,cmd)),cwd=cwd,text=True,
                           stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
        lines=[]; t0=time.time()
        for line in p.stdout:
            line=line.rstrip(); lines.append(line); print(line,flush=True)
            if timeout and (time.time()-t0)>timeout:
                p.kill(); raise subprocess.TimeoutExpired(cmd,timeout)
        rc=p.wait(); out="\n".join(lines)
        if check and rc: raise subprocess.CalledProcessError(rc,cmd,output=out)
        return rc
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode: raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p.returncode

drive.mount("/content/drive",force_remount=False)

REPO=Path("/content/SynBioCrow_GALAXY_BENCH_SEALED")
NORM=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_literature_normalized/galaxy_literature_benchmark_normalized.json")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_benchmark_sealed")
PANEL=OUT/"galaxy_benchmark_truth_stripped_panel.json"

assert NORM.is_file(),f"Missing normalized benchmark: {NORM}"
OUT.mkdir(parents=True,exist_ok=True)
if REPO.exists(): shutil.rmtree(REPO)

log("clone develop/2.3")
run(["git","clone","--depth","1","--branch","develop/2.3",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[doranet,thermo]"],timeout=1200)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
importlib.invalidate_caches()

# Syntax preflight for the scripts we are about to run.
for script in ("build_literature_benchmark_panel_2_3.py","run_literature_predictions_2_3.py"):
    run([sys.executable,"-m","py_compile",str(REPO/"scripts"/script)],timeout=120)

log("build truth-stripped 65-record benchmark panel")
run([sys.executable,str(REPO/"scripts/build_literature_benchmark_panel_2_3.py"),
     "--normalized-benchmark",str(NORM),
     "--output",str(PANEL)],cwd=REPO,timeout=300)

panel=json.loads(PANEL.read_text())
assert panel["benchmark_record_count"]==65,panel["benchmark_record_count"]
assert panel["runnable_target_count"]==64,panel["runnable_target_count"]
assert panel["excluded_target_count"]==1,panel["excluded_target_count"]
assert panel["truth_fields_in_output"] is False
print("RUNNABLE TARGETS",panel["runnable_target_count"])
print("MAPPING-LIMITED",panel["excluded_targets"])

# Freeze-policy provenance.
POLICY=REPO/"docs/SIMILARITY_POLICY_2_3.md"
policy_sha=hashlib.sha256(POLICY.read_bytes()).hexdigest()
(OUT/"frozen_similarity_policy.sha256").write_text(policy_sha+"\n")
print("FROZEN POLICY SHA256",policy_sha)

# RetroBioCat2 exact validated revision + source DB.
log("bootstrap RetroBioCat2")
RBC2_REV="c5f32561b1ed99d0701a6dbb472dc448d6c082e8"
RBC2_ROOT=Path("/content/RetroBioCat-2")
if RBC2_ROOT.exists(): shutil.rmtree(RBC2_ROOT)
run(["git","clone","https://github.com/willfinnigan/RetroBioCat-2.git",str(RBC2_ROOT)],timeout=600)
run(["git","checkout",RBC2_REV],cwd=RBC2_ROOT,timeout=300)
run([sys.executable,"-m","pip","install","-q","PyYAML","sqlalchemy","tables","gdown","tqdm","requests"],timeout=900)
run([sys.executable,"-m","pip","install","-q","--no-deps","--no-build-isolation","-e",str(RBC2_ROOT)],timeout=1200)
os.environ["PYTHONPATH"]=str(RBC2_ROOT)+os.pathsep+os.environ.get("PYTHONPATH","")
if str(RBC2_ROOT) not in sys.path: sys.path.insert(0,str(RBC2_ROOT))
importlib.invalidate_caches()

import requests
from rbc2.configs.data_path import path_to_data_folder
db=Path(path_to_data_folder)/"buyability"/"source_mols.db"
db.parent.mkdir(parents=True,exist_ok=True)
def db_ok(path):
    if not path.is_file() or path.stat().st_size<4096:return False
    try:
        con=sqlite3.connect(path)
        ok=con.execute("PRAGMA quick_check").fetchone()[0]=="ok"
        names={r[0] for r in con.execute("select name from sqlite_master where type='table'")}
        con.close()
        return ok and "building_blocks" in names
    except Exception:return False

if not db_ok(db):
    if db.exists(): db.unlink()
    for url in ["https://api.figshare.com/v2/file/download/43860240",
                "https://ndownloader.figshare.com/files/43860240"]:
        try:
            with requests.get(url,stream=True,timeout=180,allow_redirects=True) as resp:
                if resp.status_code not in (200,206): raise RuntimeError(resp.status_code)
                part=db.with_suffix(".part")
                with open(part,"wb") as fh:
                    for chunk in resp.iter_content(1024*1024):
                        if chunk: fh.write(chunk)
                part.replace(db)
            if db_ok(db): break
        except Exception as exc:
            print("RBC2 DB attempt failed",type(exc).__name__,exc,flush=True)
            if db.exists(): db.unlink()
assert db_ok(db),"RBC2 database bootstrap failed"

# RetroPath standalone bounded runtime.
log("bootstrap RetroPath standalone")
RP=Path("/content/retropath_galaxy_bench"); RP.mkdir(exist_ok=True)
BASE="https://raw.githubusercontent.com/brsynth/retropath2-wrapper/d9948aa1672873d1e3c1905a355f62f714946e67/tests/data"
rules_gz=RP/"rules_d12.csv.gz"; rules=RP/"rules_d12.csv"
sink=RP/"sink.csv"; source=RP/"source.csv"
for dest,url in {
    rules_gz:BASE+"/rules_d12.csv.gz",
    sink:BASE+"/lycopene/in/sink.csv",
    source:BASE+"/lycopene/in/source.csv"
}.items():
    urllib.request.urlretrieve(url,dest)
with gzip.open(rules_gz,"rb") as fi, rules.open("wb") as fo:
    shutil.copyfileobj(fi,fo)

RUNTIME=Path("/content/retropath_runtime_galaxy_bench")
run([sys.executable,str(REPO/"scripts/bootstrap_retropath_standalone.py"),
     "--root",str(RUNTIME),
     "--rules",str(rules),
     "--source",str(source),
     "--sink",str(sink),
     "--smoke-max-steps","2",
     "--smoke-max-structures","20"],timeout=1800)

rpm=json.loads((RUNTIME/"synbiocrow_retropath_standalone.json").read_text())
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_EXE"]=rpm["executable"]
os.environ["SYNBIOCROW_RETROPATH_RULES"]=rpm["rules_file"]
os.environ["SYNBIOCROW_RETROPATH_SINK"]=rpm["sink_file"]
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_SMOKE_PASS"]="1"
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_EQUIVALENCE_PASS"]="1"
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_MAX_STEPS"]="3"
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_TOPX"]="50"
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_TIMEOUT_MINUTES"]="10"

run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=600)
from synbiocrow import SynBioCrowEngine
ready=SynBioCrowEngine().backend_readiness()
print(json.dumps(ready,indent=2,sort_keys=True))
for bid in ("doranet","retrobiocat2","retropath_standalone"):
    assert ready.get(bid,{}).get("available"),f"{bid} unavailable: {ready.get(bid)}"

log("run held-out predictions — truth reactions are NOT passed to the runner")
run([sys.executable,str(REPO/"scripts/run_literature_predictions_2_3.py"),
     "--panel",str(PANEL),
     "--output-dir",str(OUT),
     "--max-route-steps","8",
     "--max-routes","100"],cwd=REPO,timeout=86400,stream=True)

manifest=json.loads((OUT/"sealed_manifest.json").read_text())
assert manifest["truth_accessed"] is False
assert manifest["sealed"] is True
assert manifest["target_count"]==64

bench_manifest={
    "schema":"synbiocrow.galaxy_benchmark_seal.v1",
    "benchmark_denominator":65,
    "runnable_target_count":64,
    "mapping_limited_target_count":1,
    "predictions_sha256":manifest["predictions_sha256"],
    "similarity_policy_sha256":policy_sha,
    "truth_accessed_during_prediction":False,
}
(OUT/"benchmark_seal.json").write_text(json.dumps(bench_manifest,indent=2,sort_keys=True)+"\n")

print("\nSEALED BENCHMARK PREDICTIONS SHA256 =",manifest["predictions_sha256"])
print("BENCHMARK DENOMINATOR = 65")
print("RUNNABLE TARGETS = 64")
print("TRUTH ACCESSED DURING GENERATION =",manifest["truth_accessed"])

bundle=OUT/"SynBioCrow_2_3_GALAXY_BENCHMARK_SEALED_PREDICTIONS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p!=bundle:
            z.write(p,str(p.relative_to(OUT)))
print("BUNDLE",bundle)
print("BUNDLE SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
log("COMPLETE — benchmark predictions sealed; scoring has NOT been run")
files.download(str(bundle))
